# MPLAVP Tutorial

This tutorial demonstrates how to use **MPLAVP** to retrieve aerosol vertical profiles (AVP) based on TRACER data.

- **Example 1** shows how to retrieve AVP using **ARM AMF-1** data.
- **Example 2** demonstrates retrieval using **TAMU** data.
- **Example 3** code and example for batch processing.

Typically, batch processing is used to handle large volumes of data efficiently. However, working through Example 1 and Example 2 is highly recommended, as they will help you understand how MPLAVP works and enable you to write additional code if the data source changes or adopts a new format.

Please refer to the documentation and code comments for additional information.


## Example 1 Retrieve Aerosol Vertical Profile (AVP) using the TRACER ARM AMF-1 data

MPLAVP is a procedure that consists of several main steps:
	•	Read and process MPL data, including initial corrections (e.g., dead-time, afterpulse, overlap), calculation of normalized relative backscatter (NRB), interpolation, and cloud masking.
	•	Read radiosonde data and calculate the Rayleigh backscatter profile based on pressure and temperature profiles.
	•	Perform Fernald retrieval using the MPL NRB profile and the Rayleigh backscatter profile to derive the aerosol backscatter coefficient profile.
	•	Read aerosol size distribution and concentration data, typically from SMPS and/or OPC instruments.
	•	Read or calculate the aerosol hygroscopicity parameter κ*, and use it together with the radiosonde-derived RH profile to correct the aerosol backscatter coefficient profile to dry conditions.
	•	Scale the dry aerosol backscatter coefficient profile using the total aerosol number concentration (obtained by integrating the size distribution) to produce the aerosol number concentration vertical profile.

*The hygroscopicity parameter κ can be inferred from ACSM, HTDMA, or by combining aerosol size distribution with CCN data. In this tutorial, due to data quality issues with the ARM AMF-1 CCN measurements, we demonstrate the retrieval using κ derived from TRACER ACSM data, provided by Dr. Maria Zawadowicz (DOE).

In this tutorial, we primarily call functions from the tracer_avp_processing module, which contains the main logic for each individual processing step. This module interfaces directly with the pympl, plotmpl, inversempl, and retrieval_aux modules. You will not need to interact with pympl, plotmpl, inversempl, or retrieval_aux directly during this tutorial.

The pympl, plotmpl, and inversempl modules handle the core processing and visualization of MPL data. The retrieval_aux module provides auxiliary functions for reading data from various instruments and managing supporting datasets. Some functions in retrieval_aux are legacy or intended for testing, though you may find them useful. This module also serves as a place to add future helper functions as needed.

In [ ]:
# import necessary python packages

import numpy as np
import matplotlib.pyplot as plt
import scipy
import os
import tracer_avp_processing

In [ ]:
# Initialize the data directory path. 
# Modify these paths as needed to point to your local data.

arm_mpl_dir_path                = "/../example_data/arm_mpl"
arm_mpl_calibration_dir_path    = "/../example_data/arm_mpl_calibration"
arm_radiosonde_dir_path         = "/../example_data/arm_radiosonde"
arm_sizedist_dir_path           = "/../example_data/arm_sizedist_dir_path"
arm_acsm_kappa_csv_path         = "/../example_data/arm_acsm_kappa.csv"

### Define start_time and end_time

To coordinate multiple data types from different instruments, this code uses a start_time and end_time to define the averaging period for all included datasets. For example, if the averaging window is set from 11:00 to 12:00 UTC, then only the MPL data within that hour will be used in the retrieval, and the ARM merged size distribution data will also be averaged over the same period.

An exception is the radiosonde data. If multiple radiosonde files are found within the specified time window, the process_arm_radiosonde_data function currently uses only the first file. Typically, one AVP product should be generated for each radiosonde profile. If you wish to compute an AVP over a longer averaging period that includes multiple radiosonde launches, you may use the process_arm_radiosonde_data_average function, which averages the interpolated profiles across all available soundings within the time range.


### Process MPL data

In [ ]:
# Process MPL data

armmpl_data, cloud_mask, cloud_free_column, high_snr_depol, fig1, axs1 \
                = tracer_avp_processing.process_mpl_data(start_time, end_time, ARM_mpl_folder, 
                                                         arm_ap_file, arm_ov_file, arm_dt_file,
                                                         suffix = '*.mpl',
                                                         time_resolution = 10, mov_avg_win = None,
                                                         scale_num = 30, length_treshold = 25, value_treshold = 1.5, 
                                                         edge_treshold = 1.4, contain_cloud_treshold = 1.45, cloud_pixel_number_treshold = 1,
                                                         snr_treshold = 5, not_a_cloud_treshold = 1.4,
                                                         find_cloud_range = calibration_range,
                                                         blind_zone = 0.25,
                                                         fig = None, axs = None, figsize = (15,3.5), plot_range_max = 8, 
                                                         savefig = True, showplot = True, output_folder = output_folder)